In [42]:
import json
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
from roboflow import Roboflow
from dotenv import load_dotenv

First time downloading the annotations (if annotation are already downloaded can be ignored)

In [ ]:
# securely load API Key, necessary to download the dataset
load_dotenv()
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))

project = rf.workspace("bachelorthesis").project("8-ball-pool-l530o")
dataset = project.version(3).download("coco")

loading Roboflow workspace...
loading Roboflow project...


In [60]:
def get_image_metadata(target_filename:str, annotation_path:str = './data/_annotations.coco.json'):
    # loop but there only is a train split
    splits = ["train"]
    
    for split in splits:        
        # skip if the folder/file doesn't exist
        if not os.path.exists(annotation_path):
            continue
            
        with open(annotation_path, 'r') as f:
            coco_data = json.load(f)
        
        category_map = {cat['id']: cat['name'] for cat in coco_data['categories']}

        # search for the image in this split
        image_id = None
        for img in coco_data['images']:
            if img['file_name'].startswith(target_filename) or img.get('extra', {}).get('name') == target_filename:
                image_id = img['id']
                break
        
        # found in this split, process and return
        if image_id is not None:
            image_annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == image_id and category_map[ann['category_id']] in ['Striped', 'Solid', 'Black', 'Cue']]
            
            return {
                "filename": target_filename,
                "split_found_in": split,
                "total_balls": len(image_annotations),
                "ball_list": [category_map[ann['category_id']] for ann in image_annotations],
                "raw_bboxes": [ann['bbox'] for ann in image_annotations]
            }

    # if it loops through all splits and finds nothing (should not unless)
    return f"Metadata for {target_filename} not found in train, valid, or test sets."

In [61]:
# 1 image example
prof_image = "90_png" 
metadata = get_image_metadata(target_filename=prof_image)
print(metadata)

{'filename': '90_png', 'split_found_in': 'train', 'total_balls': 9, 'ball_list': ['Striped', 'Striped', 'Solid', 'Solid', 'Black', 'Striped', 'Solid', 'Striped', 'Cue'], 'raw_bboxes': [[1288, 260, 47, 45], [842, 230, 46, 44], [667, 250, 44, 47], [639, 261, 45, 45], [839, 377, 52, 53], [625, 564, 60, 68], [683, 555, 65, 62], [660, 580, 67, 67], [800, 555, 66, 65]]}


In [54]:
def extract_table_contour(hsv_image, img_area):
    """
    Tries multiple HSV thresholds and validate the contour.
    """
    # threshold tiers
    # strict
    strict_lower = np.array([100, 150, 0])
    strict_upper = np.array([140, 255, 255])
    
    # loose
    loose_lower = np.array([95, 60, 20]) 
    loose_upper = np.array([145, 255, 255])
    
    thresholds_to_try = [
        ("Strict", strict_lower, strict_upper),
        ("Loose", loose_lower, loose_upper)
    ]
    
    # requirements to be considered a table
    MIN_AREA_RATIO = 0.125 # table must be at least 12.5% of the image
    MIN_EXTENT = 0.40     # table must fill at least 40% of its bounding box
    
    for name, lower, upper in thresholds_to_try:
        table_mask = cv2.inRange(hsv_image, lower, upper)
        contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            continue
            
        # get the largest blue contour should be the table
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        
        # --- VALIDATION CHECKS ---
        # check size
        if area < (img_area * MIN_AREA_RATIO):
            print(f"[{name} Threshold] Failed: Largest contour too small ({area} vs {img_area * MIN_AREA_RATIO})")
            continue
            
        # check shape (Extent)
        x, y, w, h = cv2.boundingRect(largest_contour)
        rect_area = w * h
        extent = float(area) / rect_area
        
        if extent < MIN_EXTENT:
            print(f"[{name} Threshold] Failed: Extent too low ({extent:.2f}). Not rectangular enough.")
            continue
            
        print(f"[{name} Threshold] Success! Table found.")


        # clean mask where only the table will be
        surface_mask = np.zeros_like(table_mask)
        cv2.drawContours(surface_mask, [largest_contour], -1, 255, -1)

        return surface_mask, table_mask
        
    # If all loops fail
    return None, None

In [55]:
def detect_balls(img_path:str, show_plots:bool=False):
  
  img = cv2.imread(img_path)
  height, width = img.shape[:2]
  img = cv2.resize(img, (800, int(800 * height / width)))

  img_area = img.shape[0] * img.shape[1]

    
  #TABLE MASK
  hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
  surface_mask, table_mask = extract_table_contour(hsv_image=hsv, img_area=img_area)

  img_test = img.copy()
  gray = cv2.cvtColor(img_test, cv2.COLOR_BGR2GRAY)
    
  gray = cv2.bitwise_and(gray, gray, mask=table_mask)
  kernel = np.ones((15, 15), np.uint8)
  eroded_mask = cv2.erode(surface_mask, kernel, iterations=1)
  masked_frame = cv2.bitwise_and(gray, gray, mask=eroded_mask)
    
  # Blur first — reduces noise, improves circle detection significantly
  blurred = cv2.GaussianBlur(masked_frame, (9, 9), 2)
    
  circles = cv2.HoughCircles(
      blurred,
      cv2.HOUGH_GRADIENT_ALT,
      dp=1.5,
      minDist=10,
      param1=300,
      param2=0.1
  )
    


  final_circles = []
  if circles is not None:
    median_test_radius = np.median(circles[0, :, 2])
    if show_plots:
      print(f"Median radius: {median_test_radius}")

      
    circles = np.round(circles[0]).astype(int)
    for x, y, r in circles:
      if((median_test_radius-5)<r<(median_test_radius+1)):

        if show_plots:
          print(f"Circle found at ({x}, {y}) with radius {r}")
          
        # draw the outer circle
        cv2.circle(img_test, (x, y), r, (0, 255, 0), 2)
        # draw the center of the circle
        cv2.circle(img_test, (x, y), 2, (0, 0, 255), 3)

        final_circles.append([x, y, r])



  if show_plots:
  
    f, axarr = plt.subplots(1, 4, figsize=(20, 10))
    axarr[0].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB))
    axarr[0].set_title("Blurred Image")
    axarr[0].axis("off")

    axarr[1].imshow(cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB))
    axarr[1].set_title("Detected Circles")
    axarr[1].axis("off")

    axarr[2].imshow(surface_mask, cmap="gray")
    axarr[2].set_title("Surface Mask")
    axarr[2].axis("off")

    axarr[3].imshow(eroded_mask, cmap="gray")
    axarr[3].set_title("Eroded Surface Mask")
    axarr[3].axis("off")

  return final_circles

In [47]:
def calculate_iou(boxA, boxB):
    # boxes folow the format: [x_min, y_min, x_max, y_max]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # compute intersection area
    interArea = max(0, xB - xA) * max(0, yB - yA)

    # compute areas of both bounding boxes
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # calculate Intersection over Union
    if float(boxAArea + boxBArea - interArea) == 0:
        return 0.0
    
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def evaluate_image_detections(truth_boxes_coco, pred_circles, scale_x, scale_y, iou_threshold=0.5):
    """
    truth_boxes_coco: list of [x, y, width, height] from Roboflow
    pred_circles: list of [x_center, y_center, radius] from HoughCircles
    scale_x, scale_y: scaling factors to adjust ground truth to the resized 800px image
    """
    
    # convert COCO truth boxes to [x_min, y_min, x_max, y_max] apply scaling due to image resizing that happens in detect_balls()
    t_boxes = []
    for x, y, w, h in truth_boxes_coco:
        x_min = x * scale_x
        y_min = y * scale_y
        x_max = (x + w) * scale_x
        y_max = (y + h) * scale_y
        t_boxes.append([x_min, y_min, x_max, y_max])

    # convert predicted circles to [x_min, y_min, x_max, y_max]
    p_boxes = []
    for cx, cy, r in pred_circles:
        p_boxes.append([cx - r, cy - r, cx + r, cy + r])

    matched_ious = []
    detected_truth_indices = set()

    # match predictions to truth boxes
    for p_box in p_boxes:
        best_iou = 0
        best_t_idx = -1
        
        for t_idx, t_box in enumerate(t_boxes):
            if t_idx in detected_truth_indices:
                continue # this ground truth ball was already claimed
            
            iou = calculate_iou(p_box, t_box)
            if iou > best_iou:
                best_iou = iou
                best_t_idx = t_idx

        # just accept balls above the threshold
        if best_iou >= iou_threshold:
            matched_ious.append(best_iou)
            detected_truth_indices.add(best_t_idx)

    true_count = len(truth_boxes_coco)
    pred_count = len(pred_circles)
    undetected_count = true_count - len(detected_truth_indices)

    return {
        "true_count": true_count,
        "pred_count": pred_count,
        "undetected_count": undetected_count,
        "matched_ious": matched_ious
    }

In [64]:
TEST_DATA_PATH = "./data/development_set/"
ims_path = os.listdir(TEST_DATA_PATH)

total_images = 0
total_absolute_error = 0
exact_count_matches = 0
total_truth_balls = 0
total_undetected_balls = 0
all_ious = []

for path in ims_path:
        
    im_name = re.findall(pattern="[0-9]+._png", string=path)[0]
    img_full_path = os.path.join(TEST_DATA_PATH, path)
    
    # get metadata/ground truth
    metadata = get_image_metadata(target_filename=im_name)
    if isinstance(metadata, str): 
        print(metadata) # Metadata not found
        continue
        
    # original image dimensions to calculate scaling factors
    original_img = cv2.imread(img_full_path)

    #error reading image
    if original_img is None:
        continue
    orig_height, orig_width = original_img.shape[:2]
    
    # dimensions  detect_balls function resizes to:
    resized_width = 800
    resized_height = int(800 * orig_height / orig_width)
    
    scale_x = resized_width / orig_width
    scale_y = resized_height / orig_height

    #-----------------DETECTION CALL-----------------
    circles = detect_balls(img_path=img_full_path, show_plots=False)
    #-----------------DETECTION CALL-----------------

    
    #-----------------Evaluation CALL-----------------
    stats = evaluate_image_detections(
        truth_boxes_coco=metadata["raw_bboxes"], 
        pred_circles=circles, 
        scale_x=scale_x, 
        scale_y=scale_y,
        iou_threshold=0.5 # could be tunned ask professor
    )
    #-----------------Evaluation CALL-----------------

    total_images += 1
    total_absolute_error += abs(stats["pred_count"] - stats["true_count"])
    
    if stats["pred_count"] == stats["true_count"]:
        exact_count_matches += 1
        
    total_truth_balls += stats["true_count"]
    total_undetected_balls += stats["undetected_count"]
    all_ious.extend(stats["matched_ious"])
    
    print(f"Processed {im_name} | True: {stats['true_count']} | Pred: {stats['pred_count']} | Undetected: {stats['undetected_count']}")

#-----------------METRICS CALCULATION-----------------
print("\n" + "="*30)
print("FINAL PIPELINE METRICS")
print("="*30)

# Ball Count MAE
mae = total_absolute_error / total_images if total_images > 0 else 0
print(f"Ball Count MAE: {mae:.2f} balls")

# Ball Count Accuracy
accuracy = (exact_count_matches / total_images) * 100 if total_images > 0 else 0
print(f"Ball Count Accuracy (Exact Match): {accuracy:.2f}%")

# BBox Detection IOU
mean_iou = np.mean(all_ious) if all_ious else 0
print(f"Average BBox IOU (True Positives): {mean_iou:.4f}")

# Average % of undetected balls (False Negatives)
undetected_pct = (total_undetected_balls / total_truth_balls) * 100 if total_truth_balls > 0 else 0
print(f"Percentage of Undetected Balls: {undetected_pct:.2f}%")


[Strict Threshold] Success! Table found.
Processed 21a_png | True: 14 | Pred: 6 | Undetected: 8
[Strict Threshold] Success! Table found.
Processed 117_png | True: 9 | Pred: 8 | Undetected: 2
[Strict Threshold] Success! Table found.
Processed 7a_png | True: 14 | Pred: 10 | Undetected: 6
[Strict Threshold] Success! Table found.
Processed 127_png | True: 16 | Pred: 6 | Undetected: 11
[Strict Threshold] Success! Table found.
Processed 32_png | True: 5 | Pred: 0 | Undetected: 5
[Strict Threshold] Success! Table found.
Processed 132_png | True: 11 | Pred: 9 | Undetected: 2
[Strict Threshold] Success! Table found.
Processed 165_png | True: 12 | Pred: 9 | Undetected: 5
[Strict Threshold] Success! Table found.
Processed 156_png | True: 5 | Pred: 4 | Undetected: 1
[Strict Threshold] Success! Table found.
Processed 91_png | True: 15 | Pred: 6 | Undetected: 9
[Strict Threshold] Failed: Largest contour too small (8369.0 vs 50000.0)
[Loose Threshold] Success! Table found.
Processed 16a_png | True: 1